# EDA de propiedades - Semana 2
**Bootcamp:** Fundamentos de Ingeniería de Datos
**Tema:** EDA (Exploratory Data Analysis) con SQL
**Dataset:** `bootcamp.bronze.properties_bronze` (datos crudos de propiedades inmobiliarias)
cargados con el notebook anterior

**Instrucciones**
1. Usar `%sql` al inicio de cada celda SQL
2. Documentar los hallazgos en comentarios
3. Subir las soluciones a GitHub

> **Importante:** un buen EDA es la base de un buen pipeline. Documentá todo lo que encuentres, ya que los hallazgos del EDA guían las transformaciones en Silver.

![image_1793276988381.png](./image_1793276988381.png "image_1793276988381.png")

![image_1793278953845.png](./image_1793278953845.png "image_1793278953845.png")

---
# PARTE 1: Exploración inicial
Dataset: `bootcamp.bronze.properties_bronze`

**Importane en bronze: dejar TODO tal cual te llega**

### Ejercicio 1.1: Verificar datos disponibles

¿Cuántos registros hay en total en la tabla `bootcamp.bronze.properties_bronze`?

*Pista: usá COUNT(*)*

In [0]:
%sql
-- Tu código acá
SELECT COUNT(*)
FROM bootcamp.bronze.properties_bronze;

-- Hallazgos:
-- 

### Ejercicio 1.2: Explorar estructura

¿Qué columnas tiene la tabla? ¿Qué tipos de datos tienen?

*Pista: usá DESCRIBE TABLE o SELECT * LIMIT 0*

In [0]:
%sql
-- Tu código acá

describe bootcamp.bronze.properties_bronze;
-- Hallazgos:
-- 

### Ejercicio 1.3: Ver muestra de datos

Mostrá las primeras 10 filas del dataset, pero solo las columnas más importantes: `id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales, antiguedad, estado, zona`.

*Pista: usá SELECT con columnas específicas y LIMIT 10*

In [0]:
%sql
-- Tu código acá
SELECT 
  id,
  ubicacion,
  precio,
  expensas,
  tipo_de_operacion,
  moneda,
  ambientes,
  metros_cuadrados_totales,
  antiguedad,
  estado,
  zona
FROM 
  bootcamp.bronze.properties_bronze
  LIMIT 10;

-- Hallazgos:
-- 

---
# PARTE 2: Análisis de valores nulos

### Ejercicio 2.1: Contar nulos por columna

¿Cuántos valores nulos hay en cada columna importante? Calculá para: `precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales, metros_cuadrados_cubiertos, orientacion_cardinal, piso, cochera, antiguedad, estado, zona`.

*Pista: usá COUNT(*) - COUNT(columna) para cada columna. Podés usar CTEs para organizar.*

In [0]:
%sql

    SELECT
        (COUNT(*) - COUNT(precio)) AS precio,
        (COUNT(*) - COUNT(expensas)) AS expensas,
        (COUNT(*) - COUNT(tipo_de_operacion)) AS tipo_de_operacion,
        (COUNT(*) - COUNT(moneda)) AS moneda,
        (COUNT(*) - COUNT(ambientes)) AS ambientes,
        (COUNT(*) - COUNT(metros_cuadrados_totales)) AS metros_cuadrados_totales,
        (COUNT(*) - COUNT(metros_cuadrados_cubiertos)) AS metros_cuadrados_cubiertos,
        (COUNT(*) - COUNT(orientacion_cardinal)) AS orientacion_cardinal,
        (COUNT(*) - COUNT(piso)) AS piso,
        (COUNT(*) - COUNT(cochera)) AS cochera,
        (COUNT(*) - COUNT(antiguedad)) AS antiguedad,
        (COUNT(*) - COUNT(estado)) AS estado,
        (COUNT(*) - COUNT(zona)) AS zona
    FROM bootcamp.bronze.properties_bronze


### Ejercicio 2.2: Porcentaje de nulos

¿Qué porcentaje de registros tienen valores nulos en las columnas clave? Calculá el porcentaje para: `precio, expensas, ambientes, metros_cuadrados_totales, metros_cuadrados_cubiertos, orientacion_cardinal, antiguedad`.

*Pista: usá ROUND((total - COUNT(columna)) * 100.0 / total, 2)*

In [0]:
%sql
WITH totales AS (
    SELECT COUNT(*) AS total
    FROM bootcamp.bronze.properties_bronze
),
pct_nulos AS (
    SELECT
        ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) AS precio,
        ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) AS expensas,
        ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) AS ambientes,
        ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) AS metros_cuadrados_totales,
        ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) AS metros_cuadrados_cubiertos,
        ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) AS orientacion_cardinal,
        ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) AS antiguedad,
        ROUND((t.total - COUNT(piso)) * 100.0 / t.total, 2) AS piso,
        ROUND((t.total - COUNT(cochera)) * 100.0 / t.total, 2) AS cochera
    FROM bootcamp.bronze.properties_bronze
    CROSS JOIN totales t
    GROUP BY t.total
)
SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
    porcentaje_nulos FOR columna IN (
        precio, expensas, ambientes,
        metros_cuadrados_totales, metros_cuadrados_cubiertos,
        orientacion_cardinal, antiguedad, piso, cochera
    )
)
WHERE porcentaje_nulos > 50
ORDER BY porcentaje_nulos DESC;

--Hallazgos: no conocía UNPIVOT que confierte columnas en FILAS (convierte, formato ancho a formato largo)

### Ejercicio 2.3: Identificar columnas críticas

¿Qué columnas tienen más del 50% de valores nulos? ¿Cuáles son críticas para el análisis?

*Pista: usá el resultado del ejercicio anterior y filtrá con HAVING o WHERE*

In [0]:
%sql
-- Tu código acá
WITH totales AS (
    SELECT COUNT(*) AS total
    FROM bootcamp.bronze.properties_bronze
),
pct_nulos AS (
    SELECT
        ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) AS precio,
        ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) AS expensas,
        ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) AS ambientes,
        ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) AS metros_cuadrados_totales,
        ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) AS metros_cuadrados_cubiertos,
        ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) AS orientacion_cardinal,
        ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) AS antiguedad,
        ROUND((t.total - COUNT(piso)) * 100.0 / t.total, 2) AS piso,
        ROUND((t.total - COUNT(cochera)) * 100.0 / t.total, 2) AS cochera
    FROM bootcamp.bronze.properties_bronze
    CROSS JOIN totales t
    GROUP BY t.total
)
SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
    porcentaje_nulos FOR columna IN (
        precio, expensas, ambientes,
        metros_cuadrados_totales, metros_cuadrados_cubiertos,
        orientacion_cardinal, antiguedad, piso, cochera
    )
)
WHERE porcentaje_nulos > 50
ORDER BY porcentaje_nulos DESC;

--Hallazgos: no conocía UNPIVOT que confierte columnas en FILAS (convierte, formato ancho a formato largo) 

---
# PARTE 3: Cardinalidad y distribución

### Ejercicio 3.1: Distribución de tipo de operación

¿Qué valores únicos hay en `tipo_de_operacion`? ¿Cuántos registros hay de cada tipo? ¿Qué porcentaje representa cada uno?

*Pista: usá GROUP BY, COUNT(*), y SUM(COUNT(*)) OVER() para calcular porcentajes*

In [0]:
%sql
-- Tu código acá
SELECT
  tipo_de_operacion,
  COUNT(*) AS total,
  ROUND( COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() , 2) AS porcentaje -- Repasar esto, cuando no hay nada ne OVER() hace que SUME el gran total
FROM bootcamp.bronze.properties_bronze
GROUP BY tipo_de_operacion
ORDER BY total DESC;
-- Hallazgos:
-- 

### Ejercicio 3.2: Distribución de moneda

¿Qué monedas hay en el dataset? ¿Cuántos registros hay de cada una? ¿Hay alguna moneda que no esperabas?

*Pista: GROUP BY moneda con conteo y porcentaje*

In [0]:
%sql
-- Tu código acá
SELECT
  moneda,
  COUNT(*) AS total,
  ROUND( COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() , 2) AS porcentaje -- Repasar esto, cuando no hay nada ne OVER() hace que SUME el gran total
FROM bootcamp.bronze.properties_bronze
GROUP BY moneda
ORDER BY total DESC;

-- Hallazgos:
-- 

### Ejercicio 3.3: Distribución de ambientes

¿Cuántas propiedades hay por cantidad de ambientes? ¿Cuál es la distribución? Ordená por cantidad de ambientes.

*Pista: GROUP BY ambientes ORDER BY ambientes*

In [0]:
%sql
-- Tu código acá
SELECT
  ambientes,
  COUNT(*) AS total,
  ROUND( COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() , 2) AS porcentaje -- Repasar esto, cuando no hay nada ne OVER() hace que SUME el gran total
FROM bootcamp.bronze.properties_bronze
GROUP BY ambientes
ORDER BY ambientes DESC;

-- Hallazgos:
-- 

### Ejercicio 3.4: Top zonas

¿Cuáles son las 15 zonas con más propiedades? Mostrá zona, cantidad y porcentaje del total.

*Pista: GROUP BY zona, COUNT(*), porcentaje, ORDER BY cantidad DESC LIMIT 15*

In [0]:
%sql

SELECT
  zona,
  COUNT(*) AS total,
  ROUND( COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() , 2) AS porcentaje -- Repasar esto, cuando no hay nada ne OVER() hace que SUME el gran total
FROM bootcamp.bronze.properties_bronze
GROUP BY zona
ORDER BY total DESC;

-- Hallazgos:
-- 

### Ejercicio 3.5: Distribución de estado

¿Qué valores hay en la columna `estado`? ¿Cuántas propiedades hay de cada estado?

*Pista: GROUP BY estado ORDER BY cantidad DESC*

In [0]:
%sql
SELECT
  estado,
  COUNT(*) AS total,
  ROUND( COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() , 2) AS porcentaje -- Repasar esto, cuando no hay nada ne OVER() hace que SUME el gran total
FROM bootcamp.bronze.properties_bronze
GROUP BY estado
ORDER BY total DESC;
-- Hallazgos:
-- 

---
# PARTE 4: Estadísticas descriptivas de variables numéricas

> **Nota sobre formato de datos:** si los datos tienen problemas de formato (strings en vez de números), primero vas a necesitar limpiarlos (por ejemplo, creando una vista temporal `propiedades_clean` que corrija los tipos antes del análisis).

In [0]:
%sql
--CREANDO LA TABLA TEMPORAL CON LOS DATOS CASTEADOS.
--la vista temporal vive hasta que se apaga el cluster. Así que sirve solamente para practicar acá. 
--Si se apaga el lcuster EJECUTAR DE VUELTA

--select * from bootcamp.bronze.properties_bronze limit 10;


CREATE OR REPLACE TEMPORARY VIEW propiedades_clean AS
SELECT
  id,
  ubicacion,
  TRY_CAST(precio AS DOUBLE)                     AS precio,
  numero,
  calle,
  TRY_CAST(expensas AS DOUBLE)                   AS expensas,
  tipo_de_operacion,
  moneda,
  TRY_CAST(ambientes AS DOUBLE)                  AS ambientes,
  TRY_CAST(metros_cuadrados_totales AS DOUBLE)   AS metros_cuadrados_totales,
  TRY_CAST(metros_cuadrados_cubiertos AS DOUBLE) AS metros_cuadrados_cubiertos,
  orientacion_cardinal,
  orientacion_inmueble,
  TRY_CAST(piso AS DOUBLE)                       AS piso,
  TRY_CAST(cochera AS DOUBLE)                    AS cochera,
  TRY_CAST(antiguedad AS DOUBLE)                 AS antiguedad,
  estado,
  tipo_vendedor,
  url,
  zona,
  fecha,
  hora
FROM bootcamp.bronze.properties_bronze;

select * from propiedades_clean limit 10


### Ejercicio 4.1: Estadísticas de precio por moneda y tipo de operación

**Objetivo:** analizar precios separados por moneda.

**Pregunta:** para cada moneda y tipo de operación, calculá: cantidad de registros, precio mínimo, máximo, promedio, mediana, percentil 25 y percentil 75. Solo incluí precios válidos (mayores a 0).

*Pista: usá MIN(), MAX(), AVG(), PERCENTILE() con GROUP BY moneda*

In [0]:
%sql
-- Tu código acá
SELECT 
    moneda,
    tipo_de_operacion,
    COUNT(*) as cantidad,
    MIN(precio) as min_precio,
    MAX(precio) as precio_maximo,
    ROUND(AVG(precio),2) as promedio_precio,
    ROUND(PERCENTILE(precio, 0.5),2) as mediana_precio,
    ROUND(PERCENTILE(precio, 0.25),2) as precio_p25,
    ROUND(PERCENTILE(precio, 0.75),2) as precio_p75
FROM propiedades_clean
WHERE precio > 0
GROUP BY moneda, tipo_de_operacion
HAVING min_precio > 1 --imposible que el precio mínimo sea 1 loco
ORDER BY cantidad DESC;
-- Hallazgos:
-- 

### Ejercicio 4.2: Estadísticas de metros cuadrados

**Objetivo:** analizar las superficies de las propiedades.

**Pregunta:** calculá estadísticas (mínimo, máximo, promedio, mediana) para `metros_cuadrados_totales` y `metros_cuadrados_cubiertos` por separado. Solo incluí valores válidos (mayores a 0).

*Pista: podés usar UNION ALL para mostrar ambas columnas en un solo resultado*

In [0]:
%sql
-- Tu código acá
-- Estadísticas de metros_cuadrados_totales
SELECT
    'metros_cuadrados_totales' AS variable,
    COUNT(*)                   AS cantidad,
    MIN(metros_cuadrados_totales)   AS minimo,
    MAX(metros_cuadrados_totales)   AS maximo,
    ROUND(AVG(metros_cuadrados_totales), 2) AS promedio,
    ROUND(PERCENTILE(metros_cuadrados_totales, 0.5), 2) AS mediana
FROM propiedades_clean
WHERE metros_cuadrados_totales > 0

UNION ALL

-- Estadísticas de metros_cuadrados_cubiertos
SELECT
    'metros_cuadrados_cubiertos' AS variable,
    COUNT(*)                    AS cantidad,
    MIN(metros_cuadrados_cubiertos)   AS minimo,
    MAX(metros_cuadrados_cubiertos)   AS maximo,
    ROUND(AVG(metros_cuadrados_cubiertos), 2) AS promedio,
    ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.5), 2) AS mediana
FROM propiedades_clean
WHERE metros_cuadrados_cubiertos > 0;

-- Hallazgos:
-- 

### Ejercicio 4.3: Análisis de antigüedad

**Objetivo:** identificar valores placeholder en antigüedad.

**Pregunta:** ¿Qué valores únicos hay en la columna `antiguedad`? ¿Cuántos registros tienen cada valor? Ordená por cantidad descendente y mostrá el top 20.

*Pista: GROUP BY antiguedad ORDER BY cantidad DESC LIMIT 20*

**Pregunta de reflexión:** ¿notás algún valor que parece ser un placeholder (como 999)?

In [0]:
%sql
-- Tu código acá
SELECT 
antiguedad,
COUNT(*) total_prop
from propiedades_clean
GROUP BY antiguedad
ORDER BY antiguedad DESC
LIMIT 10;
-- Hallazgos:
-- 

### Ejercicio 4.4: Estadísticas de antigüedad sin placeholders

**Objetivo:** calcular estadísticas reales excluyendo valores placeholder.

**Pregunta:** calculá estadísticas de antigüedad EXCLUYENDO el valor 999 (que parece ser un placeholder). Mostrá: cantidad, mínimo, máximo, promedio, mediana.

*Pista: usá WHERE antiguedad != 999 AND antiguedad IS NOT NULL AND antiguedad >= 0*

In [0]:
%sql
-- Tu código acá
SELECT 
antiguedad,
moneda,
ROUND(max(precio)) AS max_precio,
ROUND(min(precio)) AS min_precio,
ROUND(avg(precio)) AS avg_precio,
ROUND(PERCENTILE(precio, 0.5),2 ) as mediana
FROM propiedades_clean
WHERE antiguedad >0 and antiguedad != 999 and antiguedad IS NOT NULL
GROUP BY antiguedad, moneda
ORDER BY antiguedad DESC


-- Hallazgos:
-- preferí hacer el análisis con tipo de moneda, porque es muy distinto el precio en ARS que en USD

---
# PARTE 5: Detección de problemas de calidad

In [0]:
%sql
DESCRIBE propiedades_clean;

### Ejercicio 5.1: Reporte de calidad usando CTEs

Creá un reporte que muestre: total de registros, cantidad de precios inválidos (NULL o <= 0), cantidad de metros cuadrados inválidos (NULL o <= 0), cantidad de antigüedad con valor 999 (placeholder), cantidad de ambientes inválidos (NULL o 0), cantidad de moneda vacía, cantidad de tipo_operacion vacía, y el porcentaje de cada problema sobre el total.

*Pista: usá CTEs para organizar. Calculá totales y problemas por separado, luego unilos.*

In [0]:
%sql
-- Indagación para ver como son los campos que nos interesan: 
SELECT 'precio' AS campo,
       CAST(MIN(precio) AS STRING) AS minimo,
       CAST(MAX(precio) AS STRING) AS maximo
FROM propiedades_clean
UNION ALL
SELECT 'metros_cuadrados_totales',
       CAST(MIN(metros_cuadrados_totales) AS STRING),
       CAST(MAX(metros_cuadrados_totales) AS STRING)
FROM propiedades_clean
UNION ALL
SELECT 'antiguedad',
       CAST(MIN(antiguedad) AS STRING),
       CAST(MAX(antiguedad) AS STRING)
FROM propiedades_clean
UNION ALL
SELECT 'ambientes',
       CAST(MIN(ambientes) AS STRING),
       CAST(MAX(ambientes) AS STRING)
FROM propiedades_clean
UNION ALL
SELECT 'moneda',
       MIN(moneda),
       MAX(moneda)
FROM propiedades_clean
UNION ALL
SELECT 'tipo_de_operacion',
       MIN(tipo_de_operacion),
       MAX(tipo_de_operacion)
FROM propiedades_clean;

In [0]:
%sql
-- Tu código acá
%sql
WITH total as
 (SELECT 
 COUNT(*) as gran_total
 from propiedades_clean
 ),
 metricas AS  
 (
    SELECT 
    COUNT_IF(precio IS NULL OR precio <= 0) as precio_inv,
    COUNT_IF(metros_cuadrados_totales IS NULL OR metros_cuadrados_totales <= 0) as metros_inv,
    COUNT_IF(antiguedad < 0 OR antiguedad = 999 OR antiguedad IS NULL) as antiguedad_inv,
    COUNT_IF(ambientes IS NULL OR ambientes <= 0) as ambientes_inv,
    COUNT_IF(moneda IS NULL) as moneda_inv,
    COUNT_IF(tipo_de_operacion IS NULL) as tipo_de_operacion_inv
    from propiedades_clean
 )
SELECT total.gran_total, 
metricas.*,
ROUND(precio_inv*100.0/total.gran_total,2) as porc_precio_inv,
ROUND(metros_inv*100.0/total.gran_total,2) as porc_metros_inv,
ROUND(antiguedad_inv*100.0/total.gran_total,4) as porc_antiguedad_inv, --quiero ver mas exacto esta métrica sobre el total
ROUND(ambientes_inv*100.0/total.gran_total,2) as porc_ambientes_inv,
ROUND(moneda_inv*100.0/total.gran_total,2) as porc_moneda_inv,
ROUND(tipo_de_operacion_inv*100.0/total.gran_total,2) as porc_tipo_de_operacion_inv
FROM total
CROSS JOIN metricas;


%sql
SELECT
  COUNT_IF(antiguedad = 999)                AS es_999,
  COUNT_IF(antiguedad IS NULL)              AS es_nulo,
  COUNT_IF(antiguedad <= 0)                 AS es_cero_o_negativo,
  COUNT_IF(antiguedad > 0 AND antiguedad <> 999) AS es_valido,
  COUNT(*)                                  AS total
FROM propiedades_clean;

--Hallazgos: casi el 99% de los registros tiene antiguedad invalida


### Ejercicio 5.2: Detectar duplicados

¿Hay registros duplicados? Agrupá por precio y url y encontrá grupos que aparecen más de una vez. Mostrá: cantidad de grupos duplicados, total de registros duplicados, y cuántos registros «extra» hay por duplicación.

*Pista: GROUP BY precio, url HAVING COUNT(*) > 1, luego sumá los resultados*

In [0]:
%sql
-- Tu código acá
WITH grupos AS (
  SELECT precio, url, COUNT(*) AS veces
  FROM propiedades_clean
  GROUP BY precio, url
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*)           AS grupos_duplicados,
  SUM(veces)         AS registros_duplicados,
  SUM(veces - 1)     AS registros_extra
FROM grupos

-- Hallazgos:
-- 

### Ejercicio 5.3: Ver ejemplos de duplicados

Mostrá ejemplos de duplicados. Para cada grupo duplicado (precio + url), mostrá cuántas veces aparece. Ordená por cantidad de apariciones descendente y limitá a 10.

*Pista: GROUP BY precio, url HAVING COUNT(*) > 1 ORDER BY COUNT(*) DESC LIMIT 10*

In [0]:
%sql
-- Tu código acá
 SELECT 
 precio, 
 url, 
 COUNT(*) AS veces
  FROM propiedades_clean
  GROUP BY precio, url
  HAVING COUNT(*) > 1
  ORDER BY veces DESC
  LIMIT 10;

-- Hallazgos:
-- 

### Ejercicio 5.4: Detectar outliers en precio

Calculá los percentiles 1 y 99 de precio por moneda. Luego identificá propiedades con precios fuera de ese rango (outliers extremos).

*Pista: usá una CTE para calcular percentiles, luego filtrá registros fuera del rango*

In [0]:
%sql
-- Tu código acá

WITH limites AS (
    select
    moneda,
    PERCENTILE(precio, 0.01) as percentil_1,
    PERCENTILE(precio, 0.99) as percentil_99
    from propiedades_clean
    group by moneda
)
select 
    p.url, 
    p.moneda,
    p.precio, 
    l.percentil_1,
    l.percentil_99
from propiedades_clean p
join limites l
on p.moneda = l.moneda
where p.precio < l.percentil_1 or p.precio > l.percentil_99 
 
-- Hallazgos:

-- RECORDAR: un percentil N es el VALOR (un precio, en este caso) por debajo del cual
-- cae el N% de los datos, con el campo ordenado de menor a mayor.
-- Percentil 99 = el precio tal que el 99% de las propiedades cuesta eso o menos.
-- Percentil 1  = el precio tal que solo el 1% cuesta eso o menos.
-- Outliers = lo que queda por fuera: precio < percentil_1 OR precio > percentil_99

---
# PARTE 6: Análisis avanzado con Window Functions

### Ejercicio 6.1: Ranking de zonas por precio

Creá un ranking de zonas por precio promedio. Mostrá: zona, precio promedio, cantidad de propiedades, y el ranking (1 = más cara).

*Pista: calculá precio promedio por zona, luego usá ROW_NUMBER() OVER (ORDER BY precio_promedio DESC)*

In [0]:
%sql
-- Tu código acá
SELECT
zona,
AVG(precio) as precio_promedio,
COUNT(*) as cantidad_propiedades,
ROW_NUMBER() OVER (ORDER BY AVG(precio) DESC) as ranking
FROM propiedades_clean
WHERE (antiguedad > 0 OR antiguedad IS NOT NULL OR antiguedad != 999)
AND (precio >0 and precio is not null)
GROUP BY zona
ORDER BY ranking;

-- Hallazgos:
    





### Ejercicio 6.2: Comparar con promedio general

Para cada zona, mostrá: zona, precio promedio de la zona, y cómo se compara con el precio promedio general (diferencia y porcentaje).

*Pista: usá AVG(precio) OVER() para calcular el promedio general*

In [0]:

%sql

SELECT DISTINCT
    zona,
    COUNT(id) OVER (PARTITION BY zona) AS propiedades,
    ROUND(AVG(precio) OVER (), 2) AS precio_promedio_general,
    ROUND(AVG(precio) OVER (PARTITION BY zona), 2) AS precio_promedio_zona,
    ROUND(AVG(precio) OVER (PARTITION BY zona) - AVG(precio) OVER (), 2) AS diferencia,
    ROUND((AVG(precio) OVER (PARTITION BY zona) - AVG(precio) OVER ()) * 100.0 / AVG(precio) OVER (), 2) AS porcentaje_diferencia
FROM propiedades_clean;


In [0]:
%sql
WITH general AS (
  SELECT AVG(precio) AS promedio_gral
  FROM propiedades_clean
  WHERE precio > 0
),
metricas AS (
  SELECT
    zona,
    AVG(precio) AS precio_promedio_zona,
    COUNT(*)    AS cantidad_propiedades
  FROM propiedades_clean
  WHERE precio > 0
  GROUP BY zona
)
SELECT
  m.zona,
  ROUND(m.precio_promedio_zona, 2) AS precio_promedio_zona,
  m.cantidad_propiedades,
  ROUND(g.promedio_gral, 2) AS promedio_gral,
  ROUND(m.precio_promedio_zona - g.promedio_gral, 2) AS diff,
  ROUND((m.precio_promedio_zona - g.promedio_gral) * 100 / g.promedio_gral, 2) AS pct_diff
FROM metricas m
CROSS JOIN general g
ORDER BY pct_diff DESC;

### Ejercicio 6.3: Análisis temporal (si hay fecha)

Si hay una columna de fecha, agrupá por mes y mostrá: mes, cantidad de propiedades, precio promedio. ¿Hay alguna tendencia?

*Pista: usá DATE_TRUNC('month', fecha) o YEAR() y MONTH() para agrupar*

In [0]:
%sql
-- Tu código acá


-- Hallazgos:
-- 

---
# PARTE 7: Documentación de hallazgos

### Ejercicio 7.1: Resumen ejecutivo
Creá un resumen que documente:
1. Total de registros analizados
2. Principales problemas de calidad encontrados
3. Porcentaje de datos válidos vs inválidos
4. Recomendaciones para la limpieza en Silver

*Pista: combiná los resultados de ejercicios anteriores en un resumen estructurado.*

## Resumen ejecutivo (completar acá)

**1. Total de registros analizados:**
- 

**2. Principales problemas de calidad:**
- 

**3. % de datos válidos vs inválidos:**
- 

**4. Recomendaciones para Silver:**
- 

In [0]:
%sql
-- (Opcional) Query de apoyo para el resumen



---
# ✅ Checklist de finalización
- [ ] Completé la exploración inicial (Parte 1)
- [ ] Analicé valores nulos (Parte 2)
- [ ] Analicé distribuciones categóricas (Parte 3)
- [ ] Calculé estadísticas descriptivas (Parte 4)
- [ ] Identifiqué problemas de calidad (Parte 5)
- [ ] Apliqué Window Functions (Parte 6)
- [ ] Documenté mis hallazgos (Parte 7)
- [ ] Subí mi notebook a GitHub
- [ ] Creé un README explicando los hallazgos principales

**Tips:** el EDA es iterativo · documentá todo · usá CTEs · las Window Functions sirven para análisis comparativos · identificá valores placeholder (999, -1, 'N/A').

NOTAS DE CLASE: 

SI DESDE EL ORIGEN VIENE TODO HORRIBLE: 

- Borramos
- Sacamos el registro del flujo y lo llevamos a otro lado
- Soft Delete → Borramos pero sigue en origen. 
- Try_cast tener cuidado porque si me importa el dato, me lo va a traer NULO


Preguntas: 
Cómo puedo distinguir 
- Tablas basura → concat(campo_basura_1, campo_basura_2) from tabla_fea

![image_1793743490686.png](./image_1793743490686.png "image_1793743490686.png")


Como entregan en produccion un pipeline para que funcione bien en producción:
Que haya pasado los estados de dev, test, y tiempos de prueba de tests.

Trabajar en 3 entornos, idealmente dev, test/stg y luego producción. 

Test /stg: va la parte de integridad (que lo que entre debe salir)

![image_1793744938572.png](./image_1793744938572.png "image_1793744938572.png")


   
    



Preparen en cada entrevista como si fuera una charla, destacar en eso que somos buenos (comunicar bien, usar scalidraw). 

